In [2]:
from transformers import AutoImageProcessor, AutoModel
import torch
import torchvision

In [3]:
image_processor = AutoImageProcessor.from_pretrained(
    "google/efficientnet-b0", use_fast=True
)
model = AutoModel.from_pretrained("google/efficientnet-b0")

In [7]:
from pathlib import Path
from PIL import Image

def load_image(image: str | Path ):
    """
    Loads an image and processes it using the model's image processor.
    Reusable helper function for single and batch image processing.
    """
    if isinstance(image, str) or isinstance(image, Path):
        image = Image.open(image)
    elif isinstance(image, np.ndarray):
        image = Image.fromarray(image)
    elif not isinstance(image,Image.Image):
        return None

    if image.mode != "RGB":
        image = image.convert("RGB")

    image = image_processor(images=image, return_tensors="pt")

    return image["pixel_values"]

image = load_image(image="images/0HPV006.jpg")

In [9]:
import base64
from io import BytesIO

def image_to_base64(image: Image.Image) -> str:
    """
    Converts a PIL Image to a base64 encoded string.
    """
    buffered = BytesIO()
    image.save(buffered, format="JPEG")
    img_bytes = buffered.getvalue()
    img_base64 = base64.b64encode(img_bytes).decode("utf-8")
    return img_base64

# Load the image as a PIL Image (not tensor)
pil_image = Image.open("images/0HPV006.jpg")
img_base64_str = image_to_base64(pil_image)
img_base64_str

'/9j/4AAQSkZJRgABAQAAAQABAAD/2wBDAAgGBgcGBQgHBwcJCQgKDBQNDAsLDBkSEw8UHRofHh0aHBwgJC4nICIsIxwcKDcpLDAxNDQ0Hyc5PTgyPC4zNDL/2wBDAQkJCQwLDBgNDRgyIRwhMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjL/wAARCAH0AnsDASIAAhEBAxEB/8QAHwAAAQUBAQEBAQEAAAAAAAAAAAECAwQFBgcICQoL/8QAtRAAAgEDAwIEAwUFBAQAAAF9AQIDAAQRBRIhMUEGE1FhByJxFDKBkaEII0KxwRVS0fAkM2JyggkKFhcYGRolJicoKSo0NTY3ODk6Q0RFRkdISUpTVFVWV1hZWmNkZWZnaGlqc3R1dnd4eXqDhIWGh4iJipKTlJWWl5iZmqKjpKWmp6ipqrKztLW2t7i5usLDxMXGx8jJytLT1NXW19jZ2uHi4+Tl5ufo6erx8vP09fb3+Pn6/8QAHwEAAwEBAQEBAQEBAQAAAAAAAAECAwQFBgcICQoL/8QAtREAAgECBAQDBAcFBAQAAQJ3AAECAxEEBSExBhJBUQdhcRMiMoEIFEKRobHBCSMzUvAVYnLRChYkNOEl8RcYGRomJygpKjU2Nzg5OkNERUZHSElKU1RVVldYWVpjZGVmZ2hpanN0dXZ3eHl6goOEhYaHiImKkpOUlZaXmJmaoqOkpaanqKmqsrO0tba3uLm6wsPExcbHyMnK0tPU1dbX2Nna4uPk5ebn6Onq8vP09fb3+Pn6/9oADAMBAAIRAxEAPwD000GjOaOtegcAhpM0daSgBaTNFB+tAAelGaDSUwA9KSiigApKDRQAUvakHWl70AJR9aKBzQAcUGigUCAjjrSUUUwCjNHakJoASj60Zo60xB3ooozimAGkJo96Q0AA+tKaaOKD6jpQAdKCe1L2pKYCH1oJ70E0namIQ9fajOf

In [14]:
def _embed(pixel_values):
    """
    Helper function to perform model inference and return the embeddings.
    """
    with torch.no_grad():
        outputs = model(pixel_values=pixel_values)
    return outputs.pooler_output

In [15]:
output = _embed(image)  # This will run the model on the input tensor
output

tensor([[ 0.7245,  0.3867, -0.1351,  ...,  0.1646,  0.2178,  2.0449]])

torch.Size([1, 1280])